# RepairCalc: обучение taxonomy_v2
Targets: domain, activities, product category, work types и room types. Renovation stages выводятся из work types. Метрики измеряют согласие с weak labels, не независимо подтверждённую точность.

In [ ]:
!pip -q install pandas numpy scikit-learn joblib matplotlib seaborn
# CatBoost устанавливайте только если выбрали соответствующий experiment:
# !pip -q install catboost

In [ ]:
import os, subprocess
try:
 import torch; print('GPU:',torch.cuda.is_available())
except ImportError: print('PyTorch не установлен; TF-IDF не требует GPU')

In [ ]:
REPO_URL='https://github.com/OWNER/IYM.git' # замените OWNER
REPO_DIR='/content/IYM'
if not os.path.exists(REPO_DIR): subprocess.run(['git','clone',REPO_URL,REPO_DIR],check=True)
os.chdir(REPO_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# CSV можно загрузить в repo/ml_workspace/data/splits или скопировать с Drive.

In [ ]:
from pathlib import Path
import pandas as pd
from ml_workspace.ml_api import load_data, prepare_features, train, validate, evaluate, save_model, load_model, predict
from ml_workspace.ml_api.types import TrainingConfig, ProductInput
from ml_workspace.taxonomy.taxonomy_v2 import TAXONOMY_VERSION
split_dir=Path('ml_workspace/data/splits_v2')
combined=Path('/content/taxonomy_v2_combined.csv')
required=['train.csv','validation.csv','test.csv','split_manifest_reused.csv']
assert all((split_dir/x).exists() for x in required), f'Загрузите файлы: {required}'
pd.concat([pd.read_csv(split_dir/f'{s}.csv',low_memory=False) for s in ['train','validation','test']],ignore_index=True).to_csv(combined,index=False)
data=load_data(combined,split_manifest_path=split_dir/'split_manifest_reused.csv')

In [ ]:
EXPERIMENT='tfidf_logreg' # catboost_structured / embeddings_logreg (требует явной установки и загрузки encoder)
config=TrainingConfig(experiment=EXPERIMENT,use_gpu=False,max_features=80000)
features=prepare_features(data,config)

In [ ]:
model=train(features,config)
validation=validate(model,features) # thresholds выбираются только здесь
test=evaluate(model,features,'test') # test не участвует в выборе
test.metrics

In [ ]:
import matplotlib.pyplot as plt
per=pd.DataFrame(test.metrics['work_types']['per_class']).T
display(per.sort_values('support'))
per.f1.sort_values().plot.barh(figsize=(8,7),title='F1 согласия с weak labels'); plt.show()

In [ ]:
# Error analysis без переобучения/подбора по test
errors=(test.y_true!=test.y_pred).sum(axis=1)
display(data.frame[data.frame['product_id'].isin([pid for pid,e in zip(test.product_ids,errors) if e])][['product_id','name','domain_label','product_category','work_types']].head(30))

In [ ]:
artifact=Path('/content/repaircalc_taxonomy_v2_artifact')
save_model(model,artifact,validation,test)
!cp -r /content/repaircalc_taxonomy_v2_artifact /content/drive/MyDrive/

In [ ]:
loaded=load_model(artifact,TAXONOMY_VERSION,trusted=True)
examples=[ProductInput(1,'Грунтовка для стен','Грунтовки',description='Для внутренних работ'),ProductInput(2,'Семена огурца','Семена овощей')]
predict(loaded,examples)

Large Transformers автоматически не скачиваются и не обучаются. Sentence embeddings включайте осознанно после установки `sentence-transformers`; runnable default — TF-IDF + Logistic Regression.